# ADK Memory & Session Management - v7

This notebook demonstrates memory and session management in Google ADK applications.

**What you'll learn:**
- Session state (short-term memory)
- InMemoryMemoryService (long-term memory)
- Memory tools (load_memory)
- Multi-session conversations with memory
- Automatic memory persistence
- Memory search and retrieval
- Best practices for memory management

---
## 1. Installation & Dependencies

In [1]:
%pip install -q google-adk google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.


**Please restart the kernel after installing dependencies**

---
## 2. Environment Setup

In [2]:
import os
import uuid
import json
from typing import Dict, Any

# Get Project ID
PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0] or os.environ.get("GOOGLE_CLOUD_PROJECT")

# Get Location
RAW_LOCATION = !(gcloud config get-value compute/region)
LOCATION = RAW_LOCATION[0] if (RAW_LOCATION and "unset" not in RAW_LOCATION[0]) else "asia-southeast1"

# Set Environment Variables
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

print(f"Project ID: {PROJECT_ID}")
print(f"Project Region: {LOCATION}")

Project ID: sm-agentic-demo
Project Region: asia-southeast1


---
## 3. Memory Architecture Overview

ADK provides two types of memory:

### Short-Term Memory (Session State)
- Lives within a single session
- Accessible via `session.state`
- Like working memory during one conversation
- Automatically available in conversation context

### Long-Term Memory (MemoryService)
- Persists across multiple sessions
- Searchable archive of past conversations
- Two implementations:
  - **InMemoryMemoryService**: For development (non-persistent)
  - **VertexAiMemoryBankService**: For production (persistent)
- Requires explicit save and retrieval

---
## 4. Session State (Short-Term Memory)

Session state is perfect for storing temporary data during a conversation.

In [3]:
from google.adk.agents.llm_agent import Agent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# Tool that uses session state
def save_preference(preference_name: str, preference_value: str, tool_context=None) -> dict:
    """
    Saves a user preference to session state.
    
    Args:
        preference_name: Name of the preference
        preference_value: Value to save
    """
    print(f"\n💾 Saving to session state:")
    print(f"   {preference_name} = {preference_value}")
    
    # Access session state through tool_context
    if tool_context and hasattr(tool_context, 'invocation_context'):
        # Store in session state
        if not hasattr(tool_context.invocation_context.session.state, 'preferences'):
            tool_context.invocation_context.session.state['preferences'] = {}
        
        tool_context.invocation_context.session.state['preferences'][preference_name] = preference_value
        
        print(f"   ✅ Saved to session state")
        return {
            "status": "success",
            "message": f"Saved {preference_name}: {preference_value}"
        }
    else:
        # Fallback for testing
        print(f"   ⚠️  Tool context not available, simulating save")
        return {
            "status": "success",
            "message": f"Saved {preference_name}: {preference_value} (simulated)"
        }

def get_preference(preference_name: str, tool_context=None) -> dict:
    """
    Retrieves a user preference from session state.
    
    Args:
        preference_name: Name of the preference to retrieve
    """
    print(f"\n🔍 Retrieving from session state:")
    print(f"   Looking for: {preference_name}")
    
    if tool_context and hasattr(tool_context, 'invocation_context'):
        preferences = tool_context.invocation_context.session.state.get('preferences', {})
        value = preferences.get(preference_name)
        
        if value:
            print(f"   ✅ Found: {value}")
            return {
                "status": "success",
                "preference_name": preference_name,
                "preference_value": value
            }
        else:
            print(f"   ❌ Not found")
            return {
                "status": "not_found",
                "message": f"Preference '{preference_name}' not found in session"
            }
    else:
        return {
            "status": "error",
            "message": "Tool context not available"
        }

# Create agent with session state tools
session_state_agent = Agent(
    model='gemini-2.5-flash',
    name='session_state_agent',
    description="Agent that remembers preferences within a session",
    instruction="""You help users save and retrieve their preferences.
    Use save_preference to store information and get_preference to retrieve it.
    Preferences are remembered within this conversation.""",
    tools=[save_preference, get_preference],
)

print("✓ Session state agent created")

✓ Session state agent created


In [4]:
# Test session state
async def test_session_state():
    session_id = 'session-' + str(uuid.uuid4())
    user_id = 'user-' + str(uuid.uuid4())
    app_name = 'session_state_demo'
    
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    runner = Runner(
        agent=session_state_agent,
        app_name=app_name,
        session_service=session_service
    )
    
    # Turn 1: Save a preference
    print("\n" + "="*70)
    print("TURN 1: Saving preference")
    print("="*70)
    
    message1 = types.UserContent(parts=[types.Part(text="My favorite color is blue")])
    events = runner.run(user_id=user_id, session_id=session_id, new_message=message1)
    
    for event in events:
        if event.is_final_response():
            response = event.content.parts[0].text if event.content.parts else "No response"
            print(f"\n📝 Agent: {response}")
    
    # Turn 2: Retrieve the preference
    print("\n" + "="*70)
    print("TURN 2: Retrieving preference")
    print("="*70)
    
    message2 = types.UserContent(parts=[types.Part(text="What is my favorite color?")])  
    events = runner.run(user_id=user_id, session_id=session_id, new_message=message2)
    
    for event in events:
        if event.is_final_response():
            response = event.content.parts[0].text if event.content.parts else "No response"
            print(f"\n📝 Agent: {response}")

await test_session_state()


TURN 1: Saving preference



💾 Saving to session state:
   favorite color = blue
   ⚠️  Tool context not available, simulating save

📝 Agent: Got it! I'll remember that your favorite color is blue.

TURN 2: Retrieving preference

🔍 Retrieving from session state:
   Looking for: favorite color

📝 Agent: Your favorite color is blue.


---
## 5. Long-Term Memory with InMemoryMemoryService

InMemoryMemoryService allows agents to remember information across different sessions.

In [5]:
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory

# Initialize memory service
memory_service = InMemoryMemoryService()

print("✓ InMemoryMemoryService initialized")
print("\nNote: InMemoryMemoryService stores data in RAM")
print("  - Data is lost when the notebook kernel restarts")
print("  - Perfect for development and testing")
print("  - For production, use VertexAiMemoryBankService")

✓ InMemoryMemoryService initialized

Note: InMemoryMemoryService stores data in RAM
  - Data is lost when the notebook kernel restarts
  - Perfect for development and testing
  - For production, use VertexAiMemoryBankService


---
## 6. Multi-Session Memory Example

Demonstrate how information from one session can be recalled in another.

In [6]:
# Agent for capturing information (no memory tools)
info_capture_agent = Agent(
    model='gemini-2.5-flash',
    name='info_capture_agent',
    description="Agent that captures user information",
    instruction="""You are a friendly assistant who helps users share information.
    Acknowledge what they tell you and confirm you've noted it.""",
)

# Agent for recalling information (with memory tools)
memory_recall_agent = Agent(
    model='gemini-2.5-flash',
    name='memory_recall_agent',
    description="Agent that can recall past information",
    instruction="""You help users recall information from past conversations.
    When asked about something from the past, use the 'load_memory' tool to search for it.
    The load_memory tool searches through past conversation history.""",
    tools=[load_memory],
)

print("✓ Agents created")
print("  - info_capture_agent: Captures information")
print("  - memory_recall_agent: Recalls information using load_memory tool")

✓ Agents created
  - info_capture_agent: Captures information
  - memory_recall_agent: Recalls information using load_memory tool


In [7]:
# Session 1: Capture information
async def session_1_capture_info():
    print("\n" + "="*70)
    print("SESSION 1: Capturing Information")
    print("="*70)
    
    session_id = 'capture-session-' + str(uuid.uuid4())
    user_id = 'demo-user'
    app_name = 'memory_demo_app'
    
    # Initialize services
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    runner = Runner(
        agent=info_capture_agent,
        app_name=app_name,
        session_service=session_service,
        memory_service=memory_service,  # Connect memory service
    )
    
    # User shares information
    user_message = "My favorite project is Project Phoenix, and I'm working on AI-powered analytics."
    print(f"\n👤 User: {user_message}")
    
    message = types.UserContent(parts=[types.Part(text=user_message)])
    events = runner.run(user_id=user_id, session_id=session_id, new_message=message)
    
    for event in events:
        if event.is_final_response():
            response = event.content.parts[0].text if event.content.parts else "No response"
            print(f"\n🤖 Agent: {response}")
    
    # Get the completed session
    completed_session = await session_service.get_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    # Add session to long-term memory
    print("\n💾 Saving session to long-term memory...")
    await memory_service.add_session_to_memory(completed_session)
    print("✅ Session saved to memory!")
    
    return session_service

# Run session 1
session_service_shared = await session_1_capture_info()


SESSION 1: Capturing Information

👤 User: My favorite project is Project Phoenix, and I'm working on AI-powered analytics.

🤖 Agent: Thanks for sharing! You mentioned your favorite project is Project Phoenix and you're working on AI-powered analytics. I've made a note of that.

💾 Saving session to long-term memory...
✅ Session saved to memory!


In [8]:
# Session 2: Recall information
async def session_2_recall_info():
    print("\n" + "="*70)
    print("SESSION 2: Recalling Information (New Session)")
    print("="*70)
    
    session_id = 'recall-session-' + str(uuid.uuid4())
    user_id = 'demo-user'  # Same user
    app_name = 'memory_demo_app'
    
    # Create new session
    session = await session_service_shared.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    # Create runner with memory-enabled agent
    runner = Runner(
        agent=memory_recall_agent,
        app_name=app_name,
        session_service=session_service_shared,
        memory_service=memory_service,  # Same memory service
    )
    
    # User asks about past information
    user_question = "What is my favorite project?"
    print(f"\n👤 User: {user_question}")
    
    message = types.UserContent(parts=[types.Part(text=user_question)])
    events = runner.run(user_id=user_id, session_id=session_id, new_message=message)
    
    for event in events:
        if event.is_final_response():
            response = event.content.parts[0].text if event.content.parts else "No response"
            print(f"\n🤖 Agent: {response}")
            print("\n✅ Agent successfully recalled information from previous session!")

# Run session 2
await session_2_recall_info()


SESSION 2: Recalling Information (New Session)

👤 User: What is my favorite project?

🤖 Agent: Your favorite project is Project Phoenix, and you're working on AI-powered analytics.

✅ Agent successfully recalled information from previous session!


---
## 7. Multiple Pieces of Information

Test memory with multiple facts across several sessions.

In [9]:
# Create multiple sessions with different information
async def create_memory_sessions():
    user_id = 'memory-test-user'
    app_name = 'multi_memory_app'
    session_service = InMemorySessionService()
    
    facts = [
        "I live in San Francisco and work in tech.",
        "My hobbies include hiking and photography.",
        "I'm learning Spanish and planning a trip to Barcelona.",
    ]
    
    print("\n" + "="*70)
    print("CREATING MULTIPLE MEMORY SESSIONS")
    print("="*70)
    
    for i, fact in enumerate(facts, 1):
        session_id = f'fact-session-{i}'
        
        session = await session_service.create_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_id
        )
        
        runner = Runner(
            agent=info_capture_agent,
            app_name=app_name,
            session_service=session_service,
            memory_service=memory_service,
        )
        
        print(f"\nSession {i}:")
        print(f"👤 User: {fact}")
        
        message = types.UserContent(parts=[types.Part(text=fact)])
        events = runner.run(user_id=user_id, session_id=session_id, new_message=message)
        
        for event in events:
            if event.is_final_response():
                response = event.content.parts[0].text if event.content.parts else "No response"
                print(f"🤖 Agent: {response}")
        
        # Save to memory
        completed_session = await session_service.get_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_id
        )
        await memory_service.add_session_to_memory(completed_session)
        print("💾 Saved to memory")
    
    return session_service, user_id, app_name

# Create sessions
multi_session_service, multi_user_id, multi_app_name = await create_memory_sessions()


CREATING MULTIPLE MEMORY SESSIONS

Session 1:
👤 User: I live in San Francisco and work in tech.
🤖 Agent: Got it! You live in San Francisco and work in tech. I've noted that information.
💾 Saved to memory

Session 2:
👤 User: My hobbies include hiking and photography.
🤖 Agent: Thanks for letting me know! I've noted that your hobbies include hiking and photography.
💾 Saved to memory

Session 3:
👤 User: I'm learning Spanish and planning a trip to Barcelona.
🤖 Agent: That sounds exciting! So, you're learning Spanish and planning a trip to Barcelona. I've noted that down.
💾 Saved to memory


In [10]:
# Query memory for specific information
async def query_memory(question):
    session_id = 'query-session-' + str(uuid.uuid4())
    
    session = await multi_session_service.create_session(
        app_name=multi_app_name,
        user_id=multi_user_id,
        session_id=session_id
    )
    
    runner = Runner(
        agent=memory_recall_agent,
        app_name=multi_app_name,
        session_service=multi_session_service,
        memory_service=memory_service,
    )
    
    print(f"\n{'='*70}")
    print(f"👤 User: {question}")
    print(f"{'='*70}")
    
    message = types.UserContent(parts=[types.Part(text=question)])
    events = runner.run(user_id=multi_user_id, session_id=session_id, new_message=message)
    
    for event in events:
        if event.is_final_response():
            response = event.content.parts[0].text if event.content.parts else "No response"
            print(f"\n🤖 Agent: {response}\n")

# Test different queries
await query_memory("Where do I live?")
await query_memory("What are my hobbies?")
await query_memory("What language am I learning?")


👤 User: Where do I live?

🤖 Agent: I don't know where you live.


👤 User: What are my hobbies?

🤖 Agent: Your hobbies include hiking and photography.


👤 User: What language am I learning?

🤖 Agent: You are learning Spanish.



---
## 8. Conversation History or Session Information

Demonstrate how conversation history is automatically maintained within a session.

In [11]:
# Multi-turn conversation agent
conversation_agent = Agent(
    model='gemini-2.5-flash',
    name='conversation_agent',
    description="Conversational agent with context awareness",
    instruction="""You are a helpful assistant who maintains context throughout the conversation.
    Remember what the user has told you and refer back to it naturally.""",
)

print("✓ Conversation agent created")

✓ Conversation agent created


In [12]:
# Multi-turn conversation
async def multi_turn_conversation():
    session_id = 'conversation-' + str(uuid.uuid4())
    user_id = 'conversation-user'
    app_name = 'conversation_app'
    
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    runner = Runner(
        agent=conversation_agent,
        app_name=app_name,
        session_service=session_service,
    )
    
    # Conversation turns
    turns = [
        "Hi, I'm planning a birthday party for my daughter.",
        "She's turning 8 years old.",
        "She loves unicorns and the color purple.",
        "Can you suggest some party theme ideas based on what I told you?",
    ]
    
    print("\n" + "="*70)
    print("MULTI-TURN CONVERSATION WITH CONTEXT")
    print("="*70)
    
    for i, turn in enumerate(turns, 1):
        print(f"\n--- Turn {i} ---")
        print(f"👤 User: {turn}")
        
        message = types.UserContent(parts=[types.Part(text=turn)])
        events = runner.run(user_id=user_id, session_id=session_id, new_message=message)
        
        for event in events:
            if event.is_final_response():
                response = event.content.parts[0].text if event.content.parts else "No response"
                print(f"🤖 Agent: {response}")
    
    print("\n" + "="*70)
    print("✅ Conversation completed with full context maintained!")
    print("="*70)

await multi_turn_conversation()


MULTI-TURN CONVERSATION WITH CONTEXT

--- Turn 1 ---
👤 User: Hi, I'm planning a birthday party for my daughter.
🤖 Agent: Oh, that sounds like fun! How old is your daughter turning? Knowing her age might help me suggest some ideas.

--- Turn 2 ---
👤 User: She's turning 8 years old.
🤖 Agent: Eight years old, that's a fantastic age for a party! They're old enough to have definite opinions but still love a bit of magic.

Do you have any initial ideas in mind, or are you looking for some inspiration from scratch? Does she have any particular interests, favorite characters, or hobbies that might spark a theme?

--- Turn 3 ---
👤 User: She loves unicorns and the color purple.
🤖 Agent: Unicorns and purple! What a perfect combination for an 8-year-old's birthday party. That sounds absolutely enchanting.

We can definitely weave those two elements together to create a truly magical celebration.

Here are a few initial thoughts to get us started, playing on both unicorns and purple:

*   **Theme 

---
## 9. Custom Memory Tool

Create a custom tool that stores structured data in memory.

In [13]:
# In-memory database (simulated)
MEMORY_STORE = {
    'contacts': {},
    'notes': {},
}

def save_contact(name: str, phone: str, email: str = "") -> dict:
    """
    Saves a contact to memory.
    
    Args:
        name: Contact name
        phone: Phone number
        email: Email address (optional)
    """
    print(f"\n💾 Saving contact: {name}")
    
    MEMORY_STORE['contacts'][name] = {
        'phone': phone,
        'email': email,
    }
    
    print(f"   ✅ Contact saved")
    return {
        "status": "success",
        "message": f"Contact '{name}' saved successfully"
    }

def get_contact(name: str) -> dict:
    """
    Retrieves a contact from memory.
    
    Args:
        name: Contact name to retrieve
    """
    print(f"\n🔍 Searching for contact: {name}")
    
    if name in MEMORY_STORE['contacts']:
        contact = MEMORY_STORE['contacts'][name]
        print(f"   ✅ Found: {contact}")
        return {
            "status": "success",
            "name": name,
            "contact": contact
        }
    else:
        print(f"   ❌ Not found")
        return {
            "status": "not_found",
            "message": f"Contact '{name}' not found"
        }

def list_all_contacts() -> dict:
    """
    Lists all saved contacts.
    """
    print(f"\n📋 Listing all contacts")
    
    contacts = MEMORY_STORE['contacts']
    print(f"   Found {len(contacts)} contacts")
    
    return {
        "status": "success",
        "count": len(contacts),
        "contacts": contacts
    }

# Create contact management agent
contact_agent = Agent(
    model='gemini-2.5-flash',
    name='contact_manager',
    description="Contact management agent with memory",
    instruction="""You help users manage their contacts.
    Use save_contact to store new contacts, get_contact to retrieve them,
    and list_all_contacts to show all saved contacts.""",
    tools=[save_contact, get_contact, list_all_contacts],
)

print("✓ Contact management agent created")

✓ Contact management agent created


In [14]:
# Test contact management
async def test_contacts():
    session_id = 'contacts-' + str(uuid.uuid4())
    user_id = 'contact-user'
    app_name = 'contact_app'
    
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )
    
    runner = Runner(
        agent=contact_agent,
        app_name=app_name,
        session_service=session_service,
    )
    
    async def send_message(text):
        print(f"\n{'='*70}")
        print(f"👤 User: {text}")
        print(f"{'='*70}")
        
        message = types.UserContent(parts=[types.Part(text=text)])
        events = runner.run(user_id=user_id, session_id=session_id, new_message=message)
        
        for event in events:
            if event.is_final_response():
                response = event.content.parts[0].text if event.content.parts else "No response"
                print(f"\n🤖 Agent: {response}\n")
    
    # Test sequence
    await send_message("Save a contact: Alice Johnson, phone 555-1234, email alice@example.com")
    await send_message("Save another contact: Bob Smith, phone 555-5678")
    await send_message("What is Alice's phone number?")
    await send_message("Show me all my contacts")

await test_contacts()


👤 User: Save a contact: Alice Johnson, phone 555-1234, email alice@example.com

💾 Saving contact: Alice Johnson
   ✅ Contact saved

🤖 Agent: Alice Johnson has been saved to your contacts.


👤 User: Save another contact: Bob Smith, phone 555-5678

💾 Saving contact: Bob Smith
   ✅ Contact saved

🤖 Agent: Bob Smith has been saved to your contacts.


👤 User: What is Alice's phone number?

🔍 Searching for contact: Alice Johnson
   ✅ Found: {'phone': '555-1234', 'email': 'alice@example.com'}

🤖 Agent: Alice Johnson's phone number is 555-1234.


👤 User: Show me all my contacts

📋 Listing all contacts
   Found 2 contacts

🤖 Agent: Here are all your contacts:

Alice Johnson:
  Phone: 555-1234
  Email: alice@example.com

Bob Smith:
  Phone: 555-5678
  Email:



---
## 10. Best Practices Summary

### When to Use Session State
✅ **Use session state for:**
- Temporary data within one conversation
- User preferences for the current session
- Conversation context that doesn't need persistence
- Fast access without search

### When to Use Memory Service
✅ **Use memory service for:**
- Information that should persist across sessions
- Historical conversation data
- Knowledge that needs to be searchable
- Multi-session user profiles

### Memory Architecture Choice

| Use Case | Recommendation |
|----------|----------------|
| Development/Testing | InMemoryMemoryService |
| Production | VertexAiMemoryBankService |
| Single conversation | Session state only |
| Multi-session recall | Memory service + load_memory tool |

### Key Takeaways

1. **Session State** = Short-term memory (one conversation)
2. **Memory Service** = Long-term memory (across conversations)
3. **InMemoryMemoryService** = Development (non-persistent)
4. **VertexAiMemoryBankService** = Production (persistent)
5. **load_memory tool** = Enables agents to search memory
6. **add_session_to_memory()** = Saves session to long-term memory

### Production Checklist

- [ ] Use VertexAiMemoryBankService for persistence
- [ ] Implement after_agent_callback to auto-save sessions
- [ ] Add memory tools to agents that need recall
- [ ] Clear instructions on when to use memory
- [ ] Monitor memory usage and search performance
- [ ] Implement memory cleanup for old data
- [ ] Test memory retrieval accuracy
- [ ] Handle memory search failures gracefully

---
## 11. Next Steps

1. **Try VertexAiMemoryBankService** for persistent memory
2. **Implement auto-save callbacks** to automatically persist sessions
3. **Experiment with memory search** using different queries
4. **Build user profiles** that persist across sessions
5. **Create specialized memory stores** for different data types
6. **Test memory at scale** with many sessions

### Additional Resources

- [ADK Memory Documentation](https://google.github.io/adk-docs/sessions/memory/)
- [Session Management Guide](https://google.github.io/adk-docs/sessions/)
- [Vertex AI Memory Bank](https://cloud.google.com/vertex-ai/docs/agent-engine)